# Deep learning model for early warning in WWTP
# Temporal Convolutional Network (TCN)

In [ ]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score

def set_seed(seed=0):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [ ]:
data = pd.read_csv("water-treatment.data", header=None)

# Convert to numeric, coerce errors to NaN
X_raw = data.apply(pd.to_numeric, errors="coerce")

# Drop raw columns that are entirely NaN
raw_all_nan = X_raw.columns[X_raw.isna().all()]
print("Raw all-NaN columns:", list(raw_all_nan))
X_raw = X_raw.drop(columns=raw_all_nan)

# Impute remaining NaNs with column means
X_raw = X_raw.fillna(X_raw.mean())

print("Shape after cleaning:", X_raw.shape)
print("Remaining NaNs:", X_raw.isna().sum().sum())
X_raw.head()

Raw all-NaN columns: [0]
Shape after cleaning: (527, 38)
Remaining NaNs: 0


,1,2,3,4,5,6,7,8,9,10,...,29,30,31,32,33,34,35,36,37,38
0,44101.0,1.5,7.8,188.714286,407.0,166.0,66.3,4.5,2110,7.9,...,2000.0,39.085806,58.8,95.5,83.448049,70.0,89.013646,79.4,87.3,99.6
1,39024.0,3.0,7.7,188.714286,443.0,214.0,69.2,6.5,2660,7.7,...,2590.0,39.085806,60.7,94.8,83.448049,80.8,89.013646,79.5,92.1,100.0
2,32229.0,5.0,7.6,188.714286,528.0,186.0,69.9,3.4,1666,7.7,...,1888.0,39.085806,58.2,95.6,83.448049,52.9,89.013646,75.8,88.7,98.5
3,35023.0,3.5,7.9,205.000000,588.0,192.0,65.6,4.5,2430,7.8,...,1840.0,33.100000,64.2,95.3,87.300000,72.3,90.200000,82.3,89.6,100.0
4,36924.0,1.5,8.0,242.000000,496.0,176.0,64.8,4.0,2110,7.9,...,2120.0,39.085806,62.7,95.6,83.448049,71.0,92.100000,78.2,87.5,99.5


In [ ]:
daily_diff = np.linalg.norm(X_raw.diff().fillna(0).values, axis=1)
threshold = np.percentile(daily_diff, 90)
failure_day = (daily_diff > threshold).astype(int)

print("Number of failure days:", int(failure_day.sum()))

Number of failure days: 53


In [ ]:
def make_early_warning_labels(failure_series, H):
    y = np.zeros(len(failure_series), dtype=np.float32)
    for t in range(len(failure_series)):
        if t + H < len(failure_series):
            y[t] = failure_series[t+1:t+H+1].max()
    return y

y_H1 = make_early_warning_labels(failure_day, H=1)
y_H3 = make_early_warning_labels(failure_day, H=3)
y_H7 = make_early_warning_labels(failure_day, H=7)

print("Positives H=1:", int(y_H1.sum()))
print("Positives H=3:", int(y_H3.sum()))
print("Positives H=7:", int(y_H7.sum()))

Positives H=1: 53
Positives H=3: 130
Positives H=7: 237


In [ ]:
W = 14  # window length (days)

X_values = X_raw.values.astype(np.float32)
n_days, n_vars = X_values.shape

X_seq = []
idx = []

for t in range(W-1, n_days):
    X_seq.append(X_values[t-W+1:t+1])  # shape (W, C)
    idx.append(t)

X_seq = np.stack(X_seq)  # (N, W, C)
idx = np.array(idx)

print("X_seq shape:", X_seq.shape)  # (514, 14, C)

X_seq shape: (514, 14, 38)


In [ ]:
# Labels aligned to X_seq (which starts at day W-1)
y1 = y_H1[W-1:].astype(np.float32)
y3 = y_H3[W-1:].astype(np.float32)
y7 = y_H7[W-1:].astype(np.float32)

print(len(y1), X_seq.shape[0])

514 514


In [ ]:
N = X_seq.shape[0]
train_end = int(0.6 * N)
val_end   = int(0.8 * N)

X_train = X_seq[:train_end]
X_val   = X_seq[train_end:val_end]
X_test  = X_seq[val_end:]

def scale_sequences(X_train, X_val, X_test):
    # Fit scaler on TRAIN only, using all time steps
    scaler = StandardScaler()
    train_2d = X_train.reshape(-1, X_train.shape[-1])  # (N*W, C)
    scaler.fit(train_2d)

    def transform(X):
        X2 = X.reshape(-1, X.shape[-1])
        X2 = scaler.transform(X2)
        return X2.reshape(X.shape)

    return transform(X_train), transform(X_val), transform(X_test)

X_train_s, X_val_s, X_test_s = scale_sequences(X_train, X_val, X_test)

print(X_train_s.shape, X_val_s.shape, X_test_s.shape)

(308, 14, 38) (103, 14, 38) (103, 14, 38)


In [ ]:
class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)  # (N, W, C)
        self.y = torch.tensor(y, dtype=torch.float32)  # (N,)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, i):
        x = self.X[i].permute(1, 0)  # (C, W)
        y = self.y[i]
        return x, y

def make_loaders(y_all, batch_size=64):
    # y_all has length N = X_seq length
    y_train = y_all[:train_end]
    y_val   = y_all[train_end:val_end]
    y_test  = y_all[val_end:]

    train_ds = SeqDataset(X_train_s, y_train)
    val_ds   = SeqDataset(X_val_s,   y_val)
    test_ds  = SeqDataset(X_test_s,  y_test)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=False)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    test_loader  = DataLoader(test_ds, batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader

In [ ]:
class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size

    def forward(self, x):
        return x[:, :, :-self.chomp_size] if self.chomp_size > 0 else x

class TemporalBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, dilation, dropout=0.1):
        super().__init__()
        padding = (kernel_size - 1) * dilation

        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size, padding=padding, dilation=dilation)
        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.drop1 = nn.Dropout(dropout)

        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size, padding=padding, dilation=dilation)
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.drop2 = nn.Dropout(dropout)

        self.downsample = nn.Conv1d(in_ch, out_ch, 1) if in_ch != out_ch else None
        self.relu = nn.ReLU()

    def forward(self, x):
        out = self.conv1(x)
        out = self.chomp1(out)
        out = self.relu1(out)
        out = self.drop1(out)

        out = self.conv2(out)
        out = self.chomp2(out)
        out = self.relu2(out)
        out = self.drop2(out)

        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

class TCN(nn.Module):
    def __init__(self, n_inputs, channels=(32, 32, 32), kernel_size=3, dropout=0.1):
        super().__init__()
        layers = []
        in_ch = n_inputs
        for i, out_ch in enumerate(channels):
            dilation = 2 ** i
            layers.append(TemporalBlock(in_ch, out_ch, kernel_size, dilation, dropout))
            in_ch = out_ch
        self.network = nn.Sequential(*layers)
        self.head = nn.Linear(in_ch, 1)

    def forward(self, x):
        # x: (N, C, T)
        z = self.network(x)          # (N, hidden, T)
        z_last = z[:, :, -1]         # use last time step representation
        logits = self.head(z_last).squeeze(-1)  # (N,)
        return logits

In [ ]:
def train_one_model(y_all, epochs=50, lr=1e-3, batch_size=64, patience=8):
    train_loader, val_loader, test_loader = make_loaders(y_all, batch_size=batch_size)

    # pos_weight for imbalance: (#neg / #pos) on TRAIN
    y_train = y_all[:train_end]
    pos = float(y_train.sum())
    neg = float(len(y_train) - pos)
    pos_weight = torch.tensor([neg / max(pos, 1.0)], dtype=torch.float32).to(device)

    model = TCN(n_inputs=X_train_s.shape[-1], channels=(32, 32, 32), kernel_size=3, dropout=0.1).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    best_val_pr = -1.0
    best_state = None
    no_improve = 0

    for ep in range(1, epochs + 1):
        model.train()
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

        # Validate
        model.eval()
        val_probs, val_true = [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                logits = model(xb)
                probs = torch.sigmoid(logits).cpu().numpy()
                val_probs.append(probs)
                val_true.append(yb.numpy())

        val_probs = np.concatenate(val_probs)
        val_true = np.concatenate(val_true)

        val_pr = average_precision_score(val_true, val_probs) if val_true.sum() > 0 else 0.0

        if val_pr > best_val_pr + 1e-6:
            best_val_pr = val_pr
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= patience:
            break

    # Load best model
    if best_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    # Test
    model.eval()
    test_probs, test_true = [], []
    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(device)
            logits = model(xb)
            probs = torch.sigmoid(logits).cpu().numpy()
            test_probs.append(probs)
            test_true.append(yb.numpy())

    test_probs = np.concatenate(test_probs)
    test_true = np.concatenate(test_true)

    roc = roc_auc_score(test_true, test_probs) if len(np.unique(test_true)) > 1 else np.nan
    pr  = average_precision_score(test_true, test_probs) if test_true.sum() > 0 else np.nan
    y_pred = (test_probs > 0.5).astype(int)
    f1 = f1_score(test_true, y_pred) if len(np.unique(y_pred)) > 1 else 0.0

    return roc, pr, f1

In [ ]:
print("N:", N, "train_end:", train_end, "val_end:", val_end)
print("X_train_s:", X_train_s.shape, "X_val_s:", X_val_s.shape, "X_test_s:", X_test_s.shape)
print("y1 len:", len(y1), "y3 len:", len(y3), "y7 len:", len(y7))

N: 514 train_end: 308 val_end: 411
X_train_s: (308, 14, 38) X_val_s: (103, 14, 38) X_test_s: (103, 14, 38)
y1 len: 514 y3 len: 514 y7 len: 514


In [ ]:
results = {}

for H, y_all in [(1, y1), (3, y3), (7, y7)]:
    roc, pr, f1 = train_one_model(y_all, epochs=60, lr=1e-3, batch_size=64, patience=10)
    results[H] = (roc, pr, f1)
    print(f"H={H} -> ROC-AUC={roc:.3f} | PR-AUC={pr:.3f} | F1={f1:.3f}")

results

H=1 -> ROC-AUC=0.830 | PR-AUC=0.172 | F1=0.094
H=3 -> ROC-AUC=0.608 | PR-AUC=0.078 | F1=0.160
H=7 -> ROC-AUC=0.513 | PR-AUC=0.092 | F1=0.195


{1: (np.float64(0.8300000000000001), np.float64(0.17246642246642246), 0.09375),
 3: (np.float64(0.6081632653061224), np.float64(0.07764768509029628), 0.16),
 7: (np.float64(0.5130023640661938),
  np.float64(0.09205736082942373),
  0.1951219512195122)}

In [ ]:
def count_pos(y_all):
    y_test = y_all[val_end:]
    return int(y_test.sum()), len(y_test)

for H, y_all in [(1,y1),(3,y3),(7,y7)]:
    pos, n = count_pos(y_all)
    print(f"H={H}: test positives={pos} / {n} ({pos/n:.3f})")


H=1: test positives=3 / 103 (0.029)
H=3: test positives=5 / 103 (0.049)
H=7: test positives=9 / 103 (0.087)
